# Experiment A4 reproduction

This notebook checks the released A4 data and regenerates the two figures used in the paper. By default it reads the released raw results. Set `RERUN_TRAJECTORIES = True` below to rerun the full 270-case experiment from `data/A4_matrix_bank.npz`; new files are written only under `reproduced/`.

A4 is intentionally a floating-point stress experiment. Exact values at extreme conditioning can depend on the NumPy/BLAS/LAPACK environment, so a fresh full rerun need not be bit-identical to the released run.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / "data"
RAW = DATA / "raw"
FIGURES = ROOT / "figures"
REPRO = ROOT / "reproduced"
REPRO.mkdir(exist_ok=True)

RERUN_TRAJECTORIES = False

P_VALUES = (4.0, 7.3, 14.2, 32.0, 64.0, 127.3)
STRESS_LEVELS = (0, 7, 13, 20, 27, 33, 40, 47, 53)
STOP_TOL = 1e-8
UPDATE_CAP = 10_000
M, N = 100, 20

assert (DATA / "A4_matrix_bank.npz").exists()
assert (DATA / "A4_summary_by_run.csv").exists()
print("Package root:", ROOT)
print("Full trajectory rerun:", RERUN_TRAJECTORIES)

## Numerical routines

In [ ]:
def core_step(A, w, p):
    r = 1.0 - 2.0 / float(p)
    B = np.power(w, r / 2.0)[:, None] * A
    Q, R = np.linalg.qr(B, mode="reduced")
    Y = np.linalg.solve(R.T, A.T)
    X = Y.T
    w_plus = np.power(w, r) * np.sum(X * X, axis=1)
    residual = float(np.max(np.abs(np.log(w_plus / w))))
    return r, B, Q, R, Y, w_plus, residual


def diagnostics(A, B, Q, R, Y, w_plus):
    return {
        "kappa_B": float(np.linalg.cond(B, 2)),
        "norm_defect": float(abs(np.sum(w_plus) - N) / N),
        "qr_error": float(np.linalg.norm(B - Q @ R, ord="fro") / np.linalg.norm(B, ord="fro")),
        "solve_residual": float(
            np.linalg.norm(R.T @ Y - A.T, ord="fro") /
            (np.linalg.norm(R.T, 2) * np.linalg.norm(Y, "fro") + np.linalg.norm(A.T, "fro"))
        ),
    }


def run_trajectory(A, p, retain_states=False):
    w = np.full(M, N / M, dtype=np.float64)
    residuals, kappa_B, norm_defects, qr_errors, solve_residuals = [], [], [], [], []
    states = [] if retain_states else None
    raw_status = None
    failure_update = None
    failure_message = ""

    for update in range(1, UPDATE_CAP + 1):
        if retain_states:
            states.append(w.copy())
        try:
            r, B, Q, R, Y, w_plus, residual = core_step(A, w, p)
            if not (np.all(np.isfinite(w_plus)) and np.all(w_plus > 0) and np.isfinite(residual)):
                raise FloatingPointError("nonfinite/nonpositive update or residual")
        except Exception as exc:
            raw_status = "hard_failure"
            failure_update = update
            failure_message = repr(exc)
            break

        d = diagnostics(A, B, Q, R, Y, w_plus)
        residuals.append(residual)
        kappa_B.append(d["kappa_B"])
        norm_defects.append(d["norm_defect"])
        qr_errors.append(d["qr_error"])
        solve_residuals.append(d["solve_residual"])
        w = w_plus
        if residual < STOP_TOL:
            raw_status = "reported_stop"
            break

    if raw_status is None:
        raw_status = "cap_reached"

    residuals = np.asarray(residuals, dtype=float)
    def first(tol):
        idx = np.flatnonzero(residuals < tol)
        return int(idx[0] + 1) if len(idx) else -1

    return {
        "raw_status": raw_status,
        "failure_update": -1 if failure_update is None else int(failure_update),
        "failure_message": failure_message,
        "residuals": residuals,
        "kappa_B": np.asarray(kappa_B),
        "norm_defects": np.asarray(norm_defects),
        "qr_errors": np.asarray(qr_errors),
        "solve_residuals": np.asarray(solve_residuals),
        "terminal_weights": w.copy(),
        "states": None if states is None else np.asarray(states),
        "T_1e4": first(1e-4),
        "T_1e6": first(1e-6),
        "T_1e8": first(1e-8),
    }


def common_state_errors(A_ref, A_test, states, p):
    vals = []
    for w in states:
        try:
            _, _, _, _, _, ref_plus, _ = core_step(A_ref, w, p)
            _, _, _, _, _, test_plus, _ = core_step(A_test, w, p)
            e = float(np.max(np.abs(np.log(test_plus / ref_plus))))
            if not np.isfinite(e):
                raise FloatingPointError("nonfinite E_t")
            vals.append(e)
        except Exception as exc:
            return "E_diagnostic_failure", np.asarray(vals), repr(exc)
    return "ok", np.asarray(vals), ""

In [ ]:
# Clear version of the terminal reference classification.
def classify_stop(run, A_ref, p):
    if run["raw_status"] in ("cap_reached", "hard_failure"):
        return run["raw_status"], np.nan
    _, _, _, _, _, _, residual = core_step(A_ref, run["terminal_weights"], p)
    status = "reference_agrees_stop" if residual < STOP_TOL else "reference_disagrees_stop"
    return status, float(residual)

## Check the released data

In [ ]:
summary = pd.read_csv(DATA / "A4_summary_by_run.csv")
status_counts = pd.read_csv(DATA / "A4_status_counts.csv")
envelope = pd.read_csv(DATA / "A4_robustness_envelope.csv")
matrix_diag = pd.read_csv(DATA / "A4_matrix_diagnostics.csv")
map_summary = pd.read_csv(DATA / "A4_map_error_summary.csv")

assert len(summary) == 270
assert set(np.round(summary["p"], 10)) == set(P_VALUES)
assert set(summary["s"]) == set(STRESS_LEVELS)
assert summary[["base_id", "s", "p"]].duplicated().sum() == 0
assert int(status_counts[["reference_agrees_stop", "reference_disagrees_stop", "cap_reached", "hard_failure"]].to_numpy().sum()) == 270

print("Released trajectories:", len(summary))
print("Final statuses:")
print(summary["final_status"].value_counts().to_string())
print("\nRealized kappa range:", matrix_diag["realized_kappa_2"].min(), "to", matrix_diag["realized_kappa_2"].max())

In [ ]:
# Reconstruct the complete stress-level status table from the released per-run summary.
check_counts = (
    summary.groupby(["p", "s", "final_status"]).size()
    .unstack(fill_value=0)
    .reindex(columns=["reference_agrees_stop", "reference_disagrees_stop", "cap_reached", "hard_failure"], fill_value=0)
    .reset_index()
)
for c in ["reference_agrees_stop", "reference_disagrees_stop", "cap_reached", "hard_failure"]:
    assert np.array_equal(check_counts[c].to_numpy(), status_counts[c].to_numpy())
print("Status-count table reproduced exactly.")

## Optional full rerun

In [ ]:
if RERUN_TRAJECTORIES:
    reraw = REPRO / "raw"
    traj_out = reraw / "trajectories"
    map_out = reraw / "map_errors"
    traj_out.mkdir(parents=True, exist_ok=True)
    map_out.mkdir(parents=True, exist_ok=True)

    with np.load(DATA / "A4_matrix_bank.npz", allow_pickle=False) as bank:
        seeds = bank["seeds"].astype(int)
        stresses = bank["stress_levels"].astype(int)
        A_bank = bank["A_bank"].copy()

    assert A_bank.shape == (5, 9, 100, 20)
    rows = []

    def ptoken(p):
        s = f"{float(p):.10g}"
        return s.replace(".", "p").replace("-", "m")

    for j, seed in enumerate(seeds):
        base_id = j + 1
        A_ref = A_bank[j, 0]
        for p in P_VALUES:
            benign = run_trajectory(A_ref, p, retain_states=True)
            benign_status, benign_refres = classify_stop(benign, A_ref, p)
            benign_states = benign["states"]

            for si, s in enumerate(stresses):
                A = A_bank[j, si]
                run = benign if s == 0 else run_trajectory(A, p, retain_states=False)
                final_status, reference_residual = classify_stop(run, A_ref, p)
                estatus, E, emsg = common_state_errors(A_ref, A, benign_states, p)
                Emax = float(np.max(E)) if estatus == "ok" and len(E) else (0.0 if s == 0 else np.nan)

                name = f"A4_base{base_id:02d}_s{s:02d}_p{ptoken(p)}_trajectory.npz"
                np.savez_compressed(
                    traj_out / name,
                    base_id=base_id, seed=seed, s=s, p=p,
                    raw_status=run["raw_status"], final_status=final_status,
                    reference_residual=reference_residual,
                    T_1e4=run["T_1e4"], T_1e6=run["T_1e6"], T_1e8=run["T_1e8"],
                    residuals=run["residuals"], kappa_B=run["kappa_B"],
                    norm_defects=run["norm_defects"], qr_errors=run["qr_errors"],
                    solve_residuals=run["solve_residuals"], terminal_weights=run["terminal_weights"],
                    states=np.empty((0, M)) if run["states"] is None else run["states"],
                )
                mname = f"A4_base{base_id:02d}_s{s:02d}_p{ptoken(p)}_map_error.npz"
                np.savez_compressed(map_out / mname, base_id=base_id, seed=seed, s=s, p=p, status=estatus, E=E, failure_message=emsg)

                rows.append({
                    "base_id": base_id, "seed": seed, "s": s, "p": p,
                    "raw_status": run["raw_status"], "final_status": final_status,
                    "T_1e4": run["T_1e4"], "T_1e6": run["T_1e6"], "T_1e8": run["T_1e8"],
                    "reference_residual": reference_residual,
                    "E_status": estatus, "E_max": Emax,
                })
        print("finished base", base_id)

    rerun_summary = pd.DataFrame(rows)
    rerun_summary.to_csv(REPRO / "A4_summary_by_run_rerun.csv", index=False)
    print("Full rerun written under", REPRO)

## Reproduce Figure 4

In [ ]:
P_SHOW = [4, 32, 127.3]
plot_df = summary[summary["p"].isin(P_SHOW) & (summary["s"] > 0)].copy()

colors = ["#0072B2", "#CC79A7", "#009E73", "#D55E00", "#222222"]
markers = ["o", "s", "^", "D", "P"]
linestyles = ["-", "--", "-.", ":", (0, (5, 2, 1, 2))]
sizes = [8.0, 7.0, 6.0, 5.0, 4.0]

fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.2), sharey=True)
for ax, p in zip(axes, P_SHOW):
    panel = plot_df[np.isclose(plot_df["p"], p)]
    for j, base_id in enumerate(sorted(panel["base_id"].unique())):
        sub = panel[panel["base_id"] == base_id].sort_values("realized_kappa_2")
        ax.plot(sub["realized_kappa_2"], sub["E_max"], color=colors[j], linestyle=linestyles[j],
                linewidth=1.25, alpha=0.78, marker=markers[j], markersize=sizes[j],
                markerfacecolor="none", markeredgecolor=colors[j], markeredgewidth=1.25,
                label=f"Base {base_id}")
    ax.axhline(1e-8, color="#777777", linestyle="--", linewidth=0.9, alpha=0.55)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"$\kappa_2(A)$"); ax.set_title(fr"$p={p:g}$", fontsize=10)
    ax.grid(True, which="both", alpha=0.18)
axes[0].set_ylabel(r"$E_{\max}$")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=5, frameon=False, fontsize=8.1, bbox_to_anchor=(0.5, 1.04))
fig.tight_layout(rect=(0, 0, 1, 0.93))
out = REPRO / "A4_Tp_vs_p_conditioning.pdf"
fig.savefig(out, bbox_inches="tight")
plt.show()
print("saved", out)

## Reproduce Figure 5

In [ ]:
P_SHOW = [4, 32, 127.3]
S_SHOW = 33
BASES = [1, 2, 3, 4, 5]
colors = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7"]

def ptoken(p):
    return str(int(p)) if float(p).is_integer() else str(p).replace(".", "p")

def released_traj(base_id, p):
    path = RAW / "trajectories" / f"A4_base{base_id:02d}_s{S_SHOW:02d}_p{ptoken(p)}_trajectory.npz"
    with np.load(path, allow_pickle=False) as z:
        return np.asarray(z["residuals"], dtype=float)

loaded = {(p,b): released_traj(b,p) for p in P_SHOW for b in BASES}
all_positive = np.concatenate([r[r > 0] for r in loaded.values()])
ymin = min(all_positive.min(), 1e-8) / 2
ymax = all_positive.max() * 1.5
xmax = max(len(r) for r in loaded.values())

fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.2), sharex=True, sharey=True)
for ax, p in zip(axes, P_SHOW):
    for j, base_id in enumerate(BASES):
        r = loaded[(p, base_id)]
        ax.plot(np.arange(1, len(r)+1), r, color=colors[j], linewidth=0.9, alpha=0.88, label=f"Base {base_id}")
    ax.axhline(1e-8, color="#222222", linestyle="--", linewidth=1.0, alpha=0.75)
    ax.set_yscale("log"); ax.set_xlim(0, xmax); ax.set_ylim(ymin, ymax)
    ax.set_title(fr"$p={p:g}$", fontsize=10); ax.set_xlabel("Update")
    ax.grid(True, which="both", alpha=0.18)
axes[0].set_ylabel(r"$R_t=\max_i|\log(w_i^{t+1}/w_i^t)|$")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=5, frameon=False, fontsize=8.2, bbox_to_anchor=(0.5, 1.03))
fig.tight_layout(rect=(0, 0, 1, 0.92))
out = REPRO / "A4_posthoc_s33_residual_floor.pdf"
fig.savefig(out, bbox_inches="tight")
plt.show()
print("saved", out)

## Compact released-result checks

The following cell reproduces the two numerical summaries highlighted in the public experiment record: the $s=27$ stopping-time comparison and the $s=33$ residual floor.

In [ ]:
s27 = summary[summary["s"] == 27].merge(
    summary[summary["s"] == 0][["base_id", "p", "T_1e8"]].rename(columns={"T_1e8":"T_benign"}),
    on=["base_id", "p"], how="left")
s27["pct_change"] = 100 * (s27["T_1e8"] / s27["T_benign"] - 1)
print("s=27 stopping-time percentage range:", s27["pct_change"].min(), s27["pct_change"].max())

mins, tailmed = [], []
for path in sorted((RAW / "trajectories").glob("A4_base*_s33_*_trajectory.npz")):
    with np.load(path, allow_pickle=False) as z:
        r = np.asarray(z["residuals"], dtype=float)
    mins.append(r.min())
    tailmed.append(np.median(r[-1000:]))
print("s=33 whole-path minimum range:", min(mins), max(mins))
print("s=33 final-1000 median range:", min(tailmed), max(tailmed))